# 第四章：数据结构基础概念
  
---

## 4.1 导论（Introduction）

### 4.1.1 什么是数据结构？

**数据结构（Data Structure）** 是在计算机中 **组织、管理和存储数据** 的一种方式，使得数据可以被高效地访问和修改。

用一个生活中的例子来类比：

| 场景     | 数据       | 数据结构             |
| -------- | ---------- | -------------------- |
| 图书馆   | 书籍       | 按类别分区的书架系统 |
| 电话簿   | 联系人信息 | 按字母排序的索引     |
| 超市     | 商品       | 分类货架 + 标签系统  |
| 排队买票 | 顾客       | 先到先服务的队列     |

关键洞察：**相同的数据，使用不同的数据结构来组织，会导致截然不同的操作效率。** 这就是学习数据结构的核心意义。

### 4.1.2 为什么要学习数据结构？

假设你有 100 万个用户的数据需要查找某个用户：

```
方法1：逐个遍历（无序数组）
  → 最坏情况需要查找 1,000,000 次

方法2：二分查找（有序数组）
  → 最多只需要查找 约20 次（log₂(1,000,000) ≈ 20）

方法3：哈希表
    → 平均接近 1 次（均摊 O(1)，最坏情况可能退化到 O(n)）
```

**选择正确的数据结构，可以将程序的效率提升成千上万倍！**

### 4.1.3 数据结构与算法的关系

这两个概念密不可分：

```
数据结构 = 数据的组织方式（静态概念）
算法     = 操作数据的步骤（动态概念）

程序 = 数据结构 + 算法
```

用一个形象的比喻：
- **数据结构** 好比仓库的 **货架布局**（数据怎么放）
- **算法** 好比仓库的 **取货流程**（数据怎么取）

一个好的货架布局可以让取货流程更简单快速；反过来，一个复杂的取货需求可能要求你重新设计货架布局。

### 4.1.4 本章学习路线图

```
4.1 导论
 │
 ├── 4.2 & 4.3 内存模型：栈内存 vs 堆内存
 │     └── 理解数据在计算机中「住在哪里」
 │
 ├── 4.4 物理数据结构 vs 逻辑数据结构
 │     └── 理解数据结构的分类体系
 │
 ├── 4.5 抽象数据类型（ADT）
 │     └── 理解如何「设计」数据结构
 │
 └── 4.6 & 4.7 时间与空间复杂度
       └── 理解如何「评价」数据结构和算法的好坏
```

### 4.1.5 阅读与编译说明（建议先看）

为避免初学者在运行示例时踩坑，先约定以下规则：

- 本文代码默认使用 **C++20 或更新标准**（文中有 `contains` 等 C++20 写法）。
- 代码片段以“教学最小片段”为主；若你要单独编译某段示例，请补齐对应头文件。
- 文中的内存地址和图示是 **抽象示意**，用于理解机制，不代表所有平台的绝对实现细节。

---

## 4.2 栈内存与堆内存（Stack vs Heap Memory）

> 在学习数据结构之前，你必须首先理解数据在内存中是如何存储的。这是整个数据结构课程的 **地基**。

### 4.2.1 程序的内存布局

当一个 C++ 程序运行时，常见实现会把进程内存组织为若干区域：

```
┌─────────────────────────┐ 高地址
│       栈区 (Stack)        │ ← 向下增长 ↓
│                         │
├─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─┤
│                         │
│    （未使用的空间）        │
│                         │
├─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─┤
│                         │
│       堆区 (Heap)        │ ← 向上增长 ↑
│                         │
├─────────────────────────┤
│    全局/静态区 (Global)   │
├─────────────────────────┤
│      代码区 (Code)       │ ← 存放编译后的机器指令
└─────────────────────────┘ 低地址
```

> 说明：上图是常见教学模型。C++ 标准规定的是对象的**存储期和生命周期**，而不保证其物理地址布局：普通局部对象通常具有自动存储期，常由调用栈实现；`new` 得到的是动态存储期对象，通常由被称为“堆”的分配器区域提供。地址布局与增长方向均依赖平台和实现。

各区域的职责：

| 区域                             | 存储内容                     | 管理方式                             |
| -------------------------------- | ---------------------------- | ------------------------------------ |
| **代码区（Code/Text）**          | 编译后的机器指令             | 系统管理，只读                       |
| **全局/静态区（Global/Static）** | 全局变量、静态变量           | 程序启动时分配，结束时释放           |
| **动态分配区（通常称 Heap）**    | `new` 取得的动态存储         | 裸指针需由所有者 `delete`；RAII 对象可自动管理 |
| **调用栈（常见实现）**           | 自动存储期对象、调用信息     | 离开作用域时自动销毁对象             |

### 4.2.2 栈内存（Stack Memory）详解

#### 什么是栈内存？

在常见实现中，函数调用会使用 **栈帧（Stack Frame）** 保存自动存储期对象和调用信息；函数返回后相应栈帧会消失。语言层面保证的是：自动存储期对象在离开其作用域时被销毁，而具体是否使用栈帧及其布局属于实现细节。

#### 栈帧的概念

In [ ]:
#include <iostream>

In [ ]:
void functionB(int y) {
    int local_b = y * 2;    // 自动存储期；常见实现中位于 functionB 的栈帧
    std::cout << local_b << std::endl;
}   // ← local_b 离开作用域，不复存在

In [ ]:
void functionA(int x) {
    int local_a = x + 10;   // 自动存储期；常见实现中位于 functionA 的栈帧
    functionB(local_a);      // 调用 functionB，创建新的栈帧
}   // ← local_a 离开作用域，不复存在

In [ ]:
int main() {
    int a = 5;               // 自动存储期；常见实现中位于 main 的栈帧
    functionA(a);            // 调用 functionA，创建新的栈帧
    return 0;
}   // ← main 返回，a 已离开作用域

In [ ]:
main();

让我们追踪栈的变化过程：

```
=== 第1步：main() 被调用 ===

┌─────────────────┐
│ main 的栈帧      │
│   a = 5         │
└─────────────────┘ ← 栈底


=== 第2步：main() 调用 functionA(5) ===

┌─────────────────┐
│ functionA 的栈帧  │ ← 新的栈帧压入栈顶
│   x = 5         │
│   local_a = 15   │
├─────────────────┤
│ main 的栈帧      │
│   a = 5         │
└─────────────────┘


=== 第3步：functionA() 调用 functionB(15) ===

┌─────────────────┐
│ functionB 的栈帧  │ ← 又一个新的栈帧
│   y = 15        │
│   local_b = 30   │
├─────────────────┤
│ functionA 的栈帧  │
│   x = 5         │
│   local_a = 15   │
├─────────────────┤
│ main 的栈帧      │
│   a = 5         │
└─────────────────┘


=== 第4步：functionB() 执行完毕，返回 ===

┌─────────────────┐
│ functionA 的栈帧  │ ← functionB 的栈帧被弹出
│   x = 5         │
│   local_a = 15   │
├─────────────────┤
│ main 的栈帧      │
│   a = 5         │
└─────────────────┘


=== 第5步：functionA() 执行完毕，返回 ===

┌─────────────────┐
│ main 的栈帧      │ ← functionA 的栈帧也被弹出
│   a = 5         │
└─────────────────┘
```

#### 栈的工作原理——LIFO

栈遵循 **后进先出（Last In, First Out, LIFO）** 原则：
- **最后** 被调用的函数，其栈帧 **最先** 被销毁
- 就像叠盘子：你最后放上去的盘子，一定是最先被拿走的

#### 栈内存的特点

```
✅ 优点：
  1. 分配和释放极快（只需移动栈指针）
  2. 系统自动管理，不会出现内存泄漏
  3. 数据在内存中连续存放，缓存命中率高

❌ 缺点：
  1. 大小有限（通常 1MB ~ 8MB，取决于操作系统）
  2. 生命周期受限于函数作用域
  3. 无法在运行时动态决定大小（C++ 中 VLA 不是标准特性）
```

#### 栈溢出（Stack Overflow）

如果栈空间被耗尽，就会发生 **栈溢出**。最经典的例子是无穷递归：

In [ ]:
// ⚠️ 危险代码！不要运行！

In [ ]:
void infinite_recursion() {
    int large_array[1000];       // 每次调用占用约 4KB
    infinite_recursion();        // 无限递归，栈帧不断堆积
}
// 最终栈空间耗尽 → 程序崩溃（Stack Overflow）

### 4.2.3 堆内存（Heap Memory）详解

#### 什么是堆内存？

`new` 表达式会取得**动态存储期**的对象；常见实现由通常称为“堆”的分配器管理这块存储。若使用裸指针，所有者必须配对 `delete`/`delete[]`；更推荐让智能指针或容器通过 RAII 自动释放资源。

#### 在 C++ 中使用堆内存

In [ ]:
#include <iostream>

In [ ]:
int main() {
    // ===== 动态分配单个变量（常见实现中来自“堆”） =====
    int* p = new int(42);      // 返回动态对象的地址
    std::cout << *p << std::endl;  // 输出: 42
    delete p;                  // 手动释放堆内存
    p = nullptr;               // 良好习惯：释放后将指针置空

    // ===== 动态分配数组 =====
    int size = 0;
    std::cout << "请输入数组大小: ";
    std::cin >> size;          // 运行时才知道大小！
    if (!std::cin || size <= 0) {
        std::cerr << "输入非法，size 必须是正整数" << std::endl;
        return 1;
    }

    int* arr = new int[size];  // 取得 size 个 int 的动态存储
    for (int i = 0; i < size; ++i) {
        arr[i] = i * 10;
    }

    for (int i = 0; i < size; ++i) {
        std::cout << arr[i] << " ";
    }
    std::cout << std::endl;

    delete[] arr;              // 释放数组用 delete[]
    arr = nullptr;

    return 0;
}

In [ ]:
main();

#### 关键机制：自动存储期的指针与动态存储期的对象

这是理解堆内存最重要的概念之一：

In [ ]:
int* p = new int(42);

这行代码到底做了什么？

```
   常见实现：调用栈                    常见实现：动态分配区（“堆”）
    ┌─────────────┐                    ┌─────────────┐
    │ p = 0x7F00  │ ──指向──────────→  │     42      │
    │ （指针变量）  │                    │ （实际数据）  │
    └─────────────┘                    └─────────────┘
    8 字节（64位系统）                      4 字节（int）
    
    ↑ 由系统自动管理                     ↑ 由程序员手动管理
    ↑ 函数返回时自动销毁                  ↑ 必须用 delete 释放
```

若 `p` 是普通局部变量，它具有**自动存储期**，常见实现会把它放在调用栈中；
`p` 指向的 `int` 具有**动态存储期**，常见实现由“堆”分配器提供。C++ 不保证这两者的物理位置。

当函数返回时：
- `p`（自动存储期的指针）离开作用域后被销毁 ✅
- 动态分配的 `int` **不会** 因裸指针离开作用域而自动释放 ⚠️ → 如果不 `delete`，就是 **内存泄漏**

#### 堆内存的特点

```
✅ 优点：
  1. 空间大（通常可用数GB，受限于物理内存和虚拟内存）
  2. 生命周期灵活，可以跨函数使用
  3. 大小可以在运行时动态决定

❌ 缺点：
    1. 分配和释放速度比栈慢（需要内存分配器管理空闲块）
  2. 必须手动管理，容易出现内存泄漏或野指针
  3. 可能产生内存碎片
```

---

## 4.3 栈内存与堆内存——进阶对比（Stack vs Heap Continued）

### 4.3.1 栈与堆的全面对比

> 下表描述的是常见实现中的“栈/堆”对比，并非 C++ 对物理位置的保证。写 C++ 时应优先依据自动、静态或动态存储期及所有权来设计。

| 特性         | 栈（Stack）        | 堆（Heap）                 |
| ------------ | ------------------ | -------------------------- |
| **管理方式** | 系统自动管理       | 程序员手动管理             |
| **分配速度** | 极快（移动指针）   | 较慢（需要搜索空闲块）     |
| **空间大小** | 小（1~8 MB）       | 大（可达数 GB）            |
| **生命周期** | 函数作用域内       | 程序员决定                 |
| **增长方向** | 高地址 → 低地址    | 低地址 → 高地址            |
| **碎片问题** | 无                 | 有（频繁分配释放后）       |
| **访问速度** | 更快（缓存友好）   | 较慢（可能缓存不命中）     |
| **线程安全** | 每个线程有自己的栈 | 所有线程共享同一个堆       |
| **典型错误** | 栈溢出             | 内存泄漏、野指针、双重释放 |

### 4.3.2 选择存储期与所有权——决策指南

```
需要决定对象的生命周期和所有权？
│
├── 数据大小在编译时已知 且 数据量较小？
│   └── YES → 可用自动存储期的固定大小对象 ✅
│       例：int x = 10;
│           std::array<int, 100> arr;
│
├── 数据大小需要在运行时决定？
│   └── YES → 优先用拥有资源的容器/RAII 对象 ✅
│       例：std::vector<int> vec(user_input);
│
├── 数据需要在函数返回后继续存在？
│   └── YES → 返回值、容器或智能指针转移/共享所有权 ✅
│       例：返回 std::vector，或在确需动态所有权时返回 std::unique_ptr
│
└── 数据量非常大（超过几百 KB）？
    └── YES → 避免占用大量自动存储，优先用容器 ✅
        例：std::vector<int> big_array(1000000);
```

### 4.3.3 内存泄漏（Memory Leak）——堆内存最大的敌人

**内存泄漏**：程序取得了动态存储，却在不再需要时没有释放，导致这块内存无法被重新使用。

In [ ]:
// ====== 示例1：忘记 delete ======

In [ ]:
void leak_example_1() {
    int* p = new int(42);
    // ... 做一些事情 ...
    // 函数结束，p 被销毁，但 p 指向的堆内存没有被释放！
    // 这块内存就"泄漏"了——没有任何指针指向它，永远无法释放
}

// ====== 示例2：指针被覆盖 ======

In [ ]:
void leak_example_2() {
    int* p = new int(10);     // 第一块堆内存
    p = new int(20);          // p 现在指向第二块堆内存
    // 第一块堆内存（值为10）的地址丢失了！无法释放！
    delete p;                 // 只释放了第二块
}

// ====== 示例3：异常导致泄漏 ======
// 此示例故意展示异常路径上的泄漏；不要在生产代码中这样写。

In [ ]:
#include <stdexcept>

void leak_example_3() {
    int* p = new int(42);
    throw std::runtime_error("模拟异常"); // 如果这里抛出异常...
    delete p;                          // ... 这行永远不会执行！
}

### 4.3.4 现代 C++ 的解决方案：智能指针

在现代 C++（C++11 及以后）中，我们有更好的方式来管理堆内存——**智能指针**：

In [ ]:
#include <memory>
#include <iostream>

In [ ]:
void modern_cpp_example() {
    // unique_ptr：独占所有权的智能指针
    // 当 ptr 离开作用域时，它指向的堆内存会自动释放
    auto ptr = std::make_unique<int>(42);
    std::cout << *ptr << std::endl;  // 输出: 42
    // 不需要 delete！ptr 在这里被自动销毁，堆内存自动释放

    // 动态数组也可以：
    auto arr = std::make_unique<int[]>(100);
    arr[0] = 10;
    arr[1] = 20;
    // 同样不需要 delete[]
}

In [ ]:
void shared_ownership_example() {
    // shared_ptr：共享所有权的智能指针
    // 多个 shared_ptr 可以指向同一块内存
    // 当最后一个 shared_ptr 被销毁时，内存才释放
    auto sp1 = std::make_shared<int>(100);
    {
        auto sp2 = sp1;  // sp1 和 sp2 共享同一块内存
        std::cout << *sp2 << std::endl;  // 输出: 100
    }   // sp2 被销毁，但内存还在（sp1 还在用）
    
    std::cout << *sp1 << std::endl;  // 输出: 100，内存仍然有效
}   // sp1 被销毁，没有人再指向这块内存 → 自动释放

> **现代 C++ 开发准则**：尽量避免裸 `new`/`delete`，优先使用 `std::unique_ptr`、`std::shared_ptr` 以及容器（如 `std::vector`）来管理堆内存。

### 4.3.5 综合示例：从栈到堆的完整理解

In [ ]:
#include <iostream>
#include <vector>
#include <memory>
#include <string>
#include <utility>

In [ ]:
struct Student {
    std::string name;
    int age;
    
    Student(std::string n, int a) : name(std::move(n)), age(a) {
        std::cout << "构造: " << name << std::endl;
    }
    ~Student() {
        std::cout << "析构: " << name << std::endl;
    }
};

In [ ]:
void demonstrate_memory() {
    // 1. 自动存储期对象（常见实现中在调用栈上）
    int local_var = 10;           // 函数结束时自动销毁
    Student stack_student("小明", 20);  // 函数结束时自动析构
    
    // 2. 动态存储期对象（裸指针手动管理）
    Student* heap_student = new Student("小红", 21);
    delete heap_student;          // 必须手动释放
    
    // 3. 动态存储期对象（智能指针管理）
    auto smart_student = std::make_unique<Student>("小刚", 22);
    // 不需要 delete，离开作用域自动释放
    
    // 4. std::vector 管理自己的动态存储
    std::vector<int> vec = {1, 2, 3, 4, 5};
    // vec 是自动存储期对象；其元素存储由 vec 自动管理
}
// 函数结束时的析构顺序（自动存储期对象按定义的逆序销毁）：
// 析构: 小刚  ← smart_student 的析构器触发 delete
// 析构: 小明  ← stack_student 自动析构

---

## 4.4 物理数据结构与逻辑数据结构（Physical vs Logical Data Structures）

### 4.4.1 什么是物理数据结构？

**物理数据结构（Physical Data Structure）** 描述的是数据 **在内存中实际的存储方式**。它关心的是：数据在内存中是怎么排布的？

在计算机中，数据的物理存储方式只有两种基本形式：

#### （1）数组（Array）——连续存储

```
内存地址:  1000    1004    1008    1012    1016
         ┌───────┬───────┬───────┬───────┬───────┐
数据:     │  10   │  20   │  30   │  40   │  50   │
         └───────┴───────┴───────┴───────┴───────┘
         
特点：所有元素在内存中 紧密相邻、连续排列
```

**数组的核心特征**：
- 元素在内存中 **连续存储**
- 大小在创建时确定（静态数组），或通过重新分配调整（动态数组如 `std::vector`）
- 可以通过 **下标（索引）** 直接访问任意元素 → **O(1) 随机访问**
- 插入和删除可能需要移动大量元素 → **O(n)**

In [ ]:
// 固定长度原生数组（这里是自动存储期；常见实现中位于调用栈）

In [ ]:
int arr[5] = {10, 20, 30, 40, 50};
// 动态分配数组（常见实现中由“堆”提供）
int* dynamic_arr = new int[n];
// STL 动态数组（自动管理）
std::vector<int> vec = {10, 20, 30, 40, 50};

#### （2）链表（Linked List）——非连续存储

```
内存地址:  1000         2048         1520         3000
         ┌──────────┐  ┌──────────┐  ┌──────────┐  ┌──────────┐
         │ data: 10 │  │ data: 20 │  │ data: 30 │  │ data: 40 │
         │ next:2048│→ │ next:1520│→ │ next:3000│→ │ next:NULL│
         └──────────┘  └──────────┘  └──────────┘  └──────────┘
         
特点：元素散落在内存各处，通过 指针 串联起来
```

**链表的核心特征**：
- 元素在内存中 **不连续**，每个元素（节点）包含数据和指向下一个节点的指针
- 大小完全动态，随时可以增加或减少节点
- **不支持随机访问**，必须从头遍历 → **O(n)**
- 插入和删除只需修改指针 → **O(1)**（已知位置时）

In [ ]:
// 链表节点的定义

In [ ]:
struct Node {
    int data;       // 数据域
    Node* next;     // 指针域，指向下一个节点
    
    Node(int val) : data(val), next(nullptr) {}
};

In [ ]:
// 创建一个简单的链表: 10 → 20 → 30
Node* head = new Node(10);
head->next = new Node(20);
head->next->next = new Node(30);

#### 数组 vs 链表：物理结构的对比

| 特性          | 数组（Array）                    | 链表（Linked List）                |
| ------------- | -------------------------------- | ---------------------------------- |
| 内存布局      | 连续                             | 非连续                             |
| 大小          | 固定（静态）或需重新分配（动态） | 完全灵活                           |
| 随机访问      | O(1) ✅                           | O(n) ❌                             |
| 头部插入/删除 | O(n) ❌                           | O(1) ✅                             |
| 尾部插入      | O(1)（动态数组均摊）             | O(n)（无尾指针）/ O(1)（有尾指针） |
| 内存利用      | 可能浪费（预分配）               | 额外开销（指针域）                 |
| 缓存性能      | 优秀（连续内存）                 | 较差（内存碎片）                   |

### 4.4.2 什么是逻辑数据结构？

**逻辑数据结构（Logical Data Structure）** 描述的是数据之间的 **逻辑关系和操作规则**，而不关心底层具体用数组还是链表来实现。

逻辑数据结构定义的是 **"数据应该表现出什么行为"**，而不是 **"数据在内存中怎么存"**。

常见的逻辑数据结构：

#### （1）线性结构

```
元素之间是一对一的关系，像一条直线

栈（Stack）:       队列（Queue）:
  ┌───┐             ┌───┬───┬───┬───┐
  │ C │ ← 栈顶       │ A │ B │ C │ D │
  ├───┤             └───┴───┴───┴───┘
  │ B │              ↑               ↑
  ├───┤             队头(出)        队尾(入)
  │ A │ ← 栈底
  └───┘
  后进先出(LIFO)      先进先出(FIFO)
```

#### （2）非线性结构

```
树（Tree）:                    图（Graph）:
       A                      A ─── B
      / \                     |   / |
     B   C                    |  /  |
    / \   \                   | /   |
   D   E   F                  C ─── D

元素之间是一对多的关系         元素之间是多对多的关系
```

#### （3）其他逻辑结构

```
哈希表（Hash Table）:

  键(Key)    哈希函数    桶(Bucket)
  "apple" ──→ h() ──→ [0] → ("apple", 5)
  "banana"──→ h() ──→ [1] → ("banana", 3)
  "cherry"──→ h() ──→ [2] → ("cherry", 8)

通过键直接定位到值，平均 O(1)
```

### 4.4.3 物理结构与逻辑结构的关系

**核心概念**：逻辑数据结构是 **建立在** 物理数据结构之上的。任何逻辑结构的底层实现，最终都要落实到数组或链表（或两者的组合）上。

> 严谨说明：这里的“数组/链表”是教学层面的两种基础组织范式。工业实现中常见分块数组、跳表、B/B+树、哈希桶+开放寻址等形式，本质上仍可理解为“连续块组织”与“链式关联组织”的扩展或组合。

```
            逻辑数据结构                     可以用什么物理结构实现？
        ┌────────────────┐
        │   栈（Stack）   │ ───→  数组实现 ✅  /  链表实现 ✅
        ├────────────────┤
        │  队列（Queue）  │ ───→  数组实现 ✅  /  链表实现 ✅
        ├────────────────┤
        │   树（Tree）    │ ───→  数组实现 ✅  /  链表实现 ✅
        ├────────────────┤
        │   图（Graph）   │ ───→  邻接矩阵(数组) ✅ / 邻接表(链表) ✅
        ├────────────────┤
        │ 哈希表（Hash）  │ ───→  数组 + 链表（解决冲突）✅
        └────────────────┘
```

**举例**：栈的两种实现

In [ ]:
// ===== 物理实现方式1：用数组实现栈 =====
#include <stdexcept>

In [ ]:
class ArrayStack {
    int* data;
    int top_index;
    int capacity;
public:
    ArrayStack(int cap) : capacity(cap), top_index(-1) {
        data = new int[capacity];
    }
    void push(int val) {
        if (top_index >= capacity - 1) throw std::overflow_error("栈满");
        data[++top_index] = val;
    }
    int pop() {
        if (top_index < 0) throw std::underflow_error("栈空");
        return data[top_index--];
    }
    
    // 教学版手动管理资源，禁止拷贝以避免双重释放
    ArrayStack(const ArrayStack&) = delete;
    ArrayStack& operator=(const ArrayStack&) = delete;

    ~ArrayStack() { delete[] data; }
};

In [ ]:
// ===== 物理实现方式2：用链表实现栈 =====
class LinkedStack {
    struct Node {
        int data;
        Node* next;
        Node(int val, Node* nxt = nullptr) : data(val), next(nxt) {}
    };
    Node* top_node = nullptr;
public:
    LinkedStack() = default;

    // 教学版手动管理资源，禁止拷贝以避免浅拷贝后的双重释放
    LinkedStack(const LinkedStack&) = delete;
    LinkedStack& operator=(const LinkedStack&) = delete;

    void push(int val) {
        top_node = new Node(val, top_node);  // 新节点指向原栈顶
    }
    int pop() {
        if (!top_node) throw std::underflow_error("栈空");
        int val = top_node->data;
        Node* temp = top_node;
        top_node = top_node->next;
        delete temp;
        return val;
    }
    ~LinkedStack() {
        while (top_node) {
            Node* temp = top_node;
            top_node = top_node->next;
            delete temp;
        }
    }
};

**两种实现对外的行为完全一致**（都是 `push` 和 `pop`），但内部的物理存储方式不同。这就是物理结构与逻辑结构的分离。

### 4.4.4 总结图

```
┌──────────────────────────────────────────────────────┐
│                    数据结构分类                        │
├──────────────────────┬───────────────────────────────┤
│   物理数据结构         │       逻辑数据结构              │
│  （数据怎么存？）      │     （数据什么关系？）           │
│                      │                               │
│  ┌─────────┐         │  ┌──────┐  线性：栈、队列       │
│  │  数组    │         │  │ 线性  │  有序表              │
│  └─────────┘         │  └──────┘                     │
│  ┌─────────┐         │  ┌──────┐  树、图              │
│  │  链表    │         │  │ 非线性│  哈希表              │
│  └─────────┘         │  └──────┘                     │
│                      │                               │
│  ← 是实现基础 ─────── │ ─── 建立在物理结构之上 →        │
└──────────────────────┴───────────────────────────────┘
```

---

## 4.5 抽象数据类型（Abstract Data Type, ADT）

### 4.5.1 什么是抽象？

在讨论 ADT 之前，先理解 **抽象（Abstraction）** 这个概念。

抽象 = **隐藏实现细节，只暴露必要的接口**

生活中的例子：
- 你开车时，只需要知道方向盘、油门、刹车 → 这是 **接口**
- 你不需要知道发动机怎么把燃油转化为动力 → 这是被 **隐藏的实现细节**
- 汽车对你来说就是一个"抽象"：你只关心 **能做什么**，不关心 **怎么做到的**

### 4.5.2 ADT 的定义

**抽象数据类型（ADT）** = **数据（Data）** + **操作（Operations）**

ADT 只描述：
1. **存储了什么数据**（数据的逻辑含义）
2. **可以做什么操作**（操作的接口）

ADT **不描述**：
1. 数据在内存中怎么存储（用数组还是链表？）
2. 操作具体怎么实现（用什么算法？）

```
ADT 就像一份"合同"：

┌──────────────────────────────────────┐
│           ADT: 栈（Stack）            │
│                                      │
│  数据:   一组有序的元素                 │
│                                      │
│  操作:                                │
│    push(x)  → 将 x 放入栈顶           │
│    pop()    → 移除并返回栈顶元素        │
│    top()    → 返回栈顶元素（不移除）     │
│    empty()  → 栈是否为空               │
│    size()   → 返回栈中元素个数          │
│                                      │
│  规则:                                │
│    遵循后进先出（LIFO）原则             │
│                                      │
│  ⚠️ 未规定: 用数组还是链表实现          │
│  ⚠️ 未规定: push 的时间复杂度是多少     │
└──────────────────────────────────────┘
```

### 4.5.3 ADT 的详细示例

#### 示例1：ADT List（线性表）

```
ADT: List（线性表）
─────────────────────────────
数据：
  一组有序的元素集合，元素之间有前后关系

操作：
  insert(index, x)  → 在位置 index 插入元素 x
  remove(index)     → 删除位置 index 的元素
  get(index)        → 返回位置 index 的元素
  set(index, x)     → 将位置 index 的元素修改为 x
  length()          → 返回元素个数
  find(x)           → 查找元素 x 的位置
  isEmpty()         → 是否为空
  
规则：
  索引从 0 开始
  元素有明确的先后顺序
```

这个 ADT 可以有多种实现：

In [ ]:
// 实现方式1: 基于数组（ArrayList）
// → get/set 快 O(1)，insert/remove 慢 O(n)
// → C++ 中的 std::vector

// 实现方式2: 基于链表（LinkedList）
// → insert/remove 快 O(1)（已知位置时），get 慢 O(n)
// → C++ 中的 std::list

#### 示例2：ADT Queue（队列）

```
ADT: Queue（队列）
─────────────────────────────
数据：
  一组有序的元素集合

操作：
  enqueue(x)  → 将元素 x 加入队尾
  dequeue()   → 移除并返回队头元素
  front()     → 返回队头元素（不移除）
  empty()     → 队列是否为空
  size()      → 返回队列中元素个数

规则：
  遵循先进先出（FIFO）原则
  只能从队尾入队，从队头出队
```

#### 示例3：ADT Map（映射/字典）

```
ADT: Map（映射）
─────────────────────────────
数据：
  一组键值对（key-value pairs）的集合

操作：
  put(key, value)   → 插入或更新键值对
  get(key)          → 返回 key 对应的 value
  remove(key)       → 删除 key 及其对应的 value
  containsKey(key)  → 是否包含 key
  size()            → 返回键值对个数
  keys()            → 返回所有键的集合

规则：
  同一个 key 只能出现一次（唯一性）
```

可能的实现：
- 哈希表（`std::unordered_map`）→ 平均 O(1)
- 红黑树（`std::map`）→ O(log n)，但有序

### 4.5.4 ADT、数据结构、数据类型的关系

```
┌───────────────────────────────────────────────────────┐
│                                                       │
│   抽象数据类型（ADT）     "做什么"                     │
│   ┌───────────────────────────────────────┐           │
│   │  定义数据和操作的 规范/契约             │           │
│   │  例: "栈支持 push、pop、top 操作"      │           │
│   └───────────┬───────────────────────────┘           │
│               │                                       │
│               │  实现                                  │
│               ▼                                       │
│   数据结构（Data Structure）  "怎么做"                 │
│   ┌───────────────────────────────────────┐           │
│   │  ADT 的具体实现                        │           │
│   │  例: "用数组实现栈" 或 "用链表实现栈"    │           │
│   └───────────┬───────────────────────────┘           │
│               │                                       │
│               │  编码                                  │
│               ▼                                       │
│   代码（Code）                                        │
│   ┌───────────────────────────────────────┐           │
│   │  用具体编程语言写出的程序              │           │
│   │  例: C++ 的 std::stack<int>           │           │
│   └───────────────────────────────────────┘           │
│                                                       │
└───────────────────────────────────────────────────────┘
```

### 4.5.5 为什么 ADT 很重要？

ADT 体现了 **面向接口编程** 的思想：

In [ ]:
#include <stack>
#include <queue>

// 使用者只需要知道 ADT 的接口，不需要知道内部实现

In [ ]:
void process_tasks() {
    std::stack<int> task_stack;      // 我只知道它是一个栈
    task_stack.push(1);             // 我只知道可以 push
    task_stack.push(2);
    int top = task_stack.top();     // 我只知道可以看栈顶
    task_stack.pop();               // 我只知道可以 pop
    
    // 我不需要知道：
    // - 底层用了 deque 还是 vector？
    // - push 时有没有扩容？
    // - 内存是怎么分配的？
    
    // 正是因为 ADT 的抽象，我可以专注于解决业务问题
    // 而不是纠结于底层实现细节
}

**好处总结**：
1. **降低复杂性**：使用者不需要理解内部实现
2. **提高可维护性**：可以更换底层实现而不影响使用者
3. **促进团队协作**：不同人可以并行开发接口和实现

---

## 4.6 时间复杂度与空间复杂度（Time and Space Complexity）

### 4.6.1 为什么需要复杂度分析？

假设你和同事分别写了一个排序程序：
- 你的程序在 10 万个数据时需要 **0.1 秒**
- 同事的程序需要 **0.3 秒**

能说你的程序更好吗？**不一定！** 因为：
- 你用的电脑可能更快
- 你的测试数据可能更"友好"
- 数据量变成 1000 万时，结果可能完全反转

**我们需要一种与硬件无关、与具体数据无关、只描述"增长趋势"的方式来衡量算法的效率。** 这就是 **复杂度分析**。

### 4.6.2 时间复杂度（Time Complexity）

#### 核心思想

时间复杂度衡量的不是程序的"运行时间"，而是 **当输入规模增大时，执行的基本操作次数增长得有多快**。

#### 大 O 表示法（Big-O Notation）

大 O 表示法描述的是算法增长速率的**渐近上界**。它常用来给出最坏情况上界，但“大 O”本身并不只表示最坏情况；平均或摊还分析也可以用大 O 表达。

```
T(n) = 3n² + 5n + 10

当 n 很大时：
  3n²    → 支配项（增长最快）
  5n     → 可以忽略
  10     → 可以忽略
  系数3  → 也可以忽略（因为我们关心的是"增长趋势"，不是具体值）

因此 T(n) = O(n²)
```

**大 O 的简化规则**：
1. **只保留最高阶项**：O(n³ + n² + n) = O(n³)
2. **去掉常数系数**：O(3n²) = O(n²)
3. **常数操作是 O(1)**：O(5) = O(1)

#### 常见时间复杂度——从快到慢

```
O(1) < O(log n) < O(n) < O(n log n) < O(n²) < O(n³) < O(2ⁿ) < O(n!)
```

以 n = 1,000,000 为例，各复杂度的操作次数：

| 复杂度         | 名称     | n = 10⁶ 时的操作次数 | 直观感受         |
| -------------- | -------- | -------------------- | ---------------- |
| **O(1)**       | 常数     | 1                    | 瞬间             |
| **O(log n)**   | 对数     | ≈ 20                 | 瞬间             |
| **O(n)**       | 线性     | 1,000,000            | 很快             |
| **O(n log n)** | 线性对数 | ≈ 20,000,000         | 快               |
| **O(n²)**      | 平方     | 1,000,000,000,000    | 很慢             |
| **O(n³)**      | 立方     | 10¹⁸                 | 不可接受         |
| **O(2ⁿ)**      | 指数     | 天文数字             | 宇宙毁灭也算不完 |

#### 直觉上理解：增长曲线

```
操作次数
    │
    │                                    / O(2ⁿ)
    │                                   /
    │                              ____/ O(n²)
    │                         ___/
    │                     __/
    │                 __/   __________ O(n log n)
    │             __/  ___/
    │          _/  __/
    │        / __/    ________________ O(n)
    │      /__/
    │    _/   ________________________ O(log n)
    │  _/
    │_/ ______________________________ O(1)
    └──────────────────────────────────→ 输入规模 n
```

#### 各复杂度详解

**O(1) —— 常数时间**

无论输入多大，操作次数都不变：

In [ ]:
// 前提：vec 非空

In [ ]:
int get_first(const std::vector<int>& vec) {
    return vec[0];    // 不管 vec 有多少元素，这永远是 1 次操作
}

// 前提：arr 非空，且 index 在调用者持有的数组范围内

In [ ]:
int get_element(const int arr[], int index) {
    return arr[index]; // 数组随机访问，永远是 O(1)
}

**O(log n) —— 对数时间**

每一步都将问题规模缩小一半：

In [ ]:
// 二分查找：在有序数组中查找目标值

In [ ]:
int binary_search(const std::vector<int>& sorted_arr, int target) {
    int left = 0, right = sorted_arr.size() - 1;
    
    while (left <= right) {
        int mid = left + (right - left) / 2;
        if (sorted_arr[mid] == target) return mid;
        else if (sorted_arr[mid] < target) left = mid + 1;
        else right = mid - 1;
    }
    return -1;  // 未找到
}
// 每次循环，搜索范围缩小一半
// n → n/2 → n/4 → n/8 → ... → 1
// 需要 log₂(n) 步

**O(n) —— 线性时间**

操作次数与输入规模成正比：

In [ ]:
// 在无序数组中查找目标值

In [ ]:
int linear_search(const std::vector<int>& arr, int target) {
    for (int i = 0; i < arr.size(); ++i) {  // 最坏情况遍历整个数组
        if (arr[i] == target) return i;
    }
    return -1;
}

**O(n log n) —— 线性对数时间**

典型的高效排序算法：

In [ ]:
// 归并排序的时间复杂度为 O(n log n)
// 直觉：将数组分成两半（log n 层），每层做 O(n) 的合并工作
// 总计: O(n) × O(log n) = O(n log n)

In [ ]:
std::vector<int> vec = {5, 3, 8, 1, 9, 2};
std::sort(vec.begin(), vec.end());
// STL sort 平均 O(n log n)

**O(n²) —— 平方时间**

嵌套循环的典型复杂度：

In [ ]:
// 冒泡排序

In [ ]:
void bubble_sort(std::vector<int>& arr) {
    int n = arr.size();
    for (int i = 0; i < n - 1; ++i) {         // 外层 n 次
        for (int j = 0; j < n - 1 - i; ++j) { // 内层 n 次
            if (arr[j] > arr[j + 1]) {
                std::swap(arr[j], arr[j + 1]);
            }
        }
    }
}
// 总操作次数 ≈ n × n = n²

### 4.6.3 空间复杂度（Space Complexity）

空间复杂度衡量的是 **算法运行时额外需要的内存空间** 随输入规模增长的速率。

> 注意：空间复杂度通常只计算 **额外空间**（辅助空间），不包括输入数据本身占用的空间。

#### 常见的空间复杂度

In [ ]:
// ===== O(1) 空间 =====
// 只使用了固定数量的额外变量

In [ ]:
void swap(int& a, int& b) {
    int temp = a;   // 只用了一个额外变量 temp
    a = b;
    b = temp;
}

// ===== O(n) 空间 =====
// 额外空间与输入规模成正比

In [ ]:
std::vector<int> copy_array(const std::vector<int>& arr) {
    std::vector<int> result(arr.size());   // 创建了一个与输入同样大的数组
    for (size_t i = 0; i < arr.size(); ++i) {
        result[i] = arr[i];
    }
    return result;
}

// ===== O(n) 空间（递归调用栈） =====
// 递归深度为 n，每层占用栈帧空间

In [ ]:
int factorial(int n) {
    if (n <= 1) return 1;
    return n * factorial(n - 1);
    // 递归调用栈：
    // factorial(5) 等待 factorial(4) 等待 factorial(3) ...
    // 同时存在 n 个栈帧 → O(n) 空间
}

// ===== O(n²) 空间 =====
// 创建了二维数组

In [ ]:
void create_matrix(int n) {
    std::vector<std::vector<int>> matrix(n, std::vector<int>(n, 0));
    // n × n 的矩阵 → O(n²) 空间
}

### 4.6.4 时间与空间的权衡（Trade-off）

在算法设计中，经常需要在时间和空间之间做权衡：

```
用更多空间 → 换取更少的时间（空间换时间）
用更少空间 → 可能需要更多的时间（时间换空间）
```

**经典案例：两数之和问题**

给定一个数组和一个目标值，找出两个相加等于目标值的元素。

In [ ]:
#include <unordered_map>
#include <utility>
#include <vector>

// 方案1: 暴力法 —— 时间 O(n²)，空间 O(1)

In [ ]:
std::pair<int,int> two_sum_brute(const std::vector<int>& nums, int target) {
    for (int i = 0; i < nums.size(); ++i) {
        for (int j = i + 1; j < nums.size(); ++j) {
            if (static_cast<long long>(nums[i]) + nums[j] == target) {
                return {i, j};
            }
        }
    }
    return {-1, -1};
}

// 方案2: 哈希表法 —— 期望/平均时间 O(n)，最坏 O(n²)，空间 O(n)

In [ ]:
std::pair<int,int> two_sum_hash(const std::vector<int>& nums, int target) {
    std::unordered_map<long long, int> seen;   // 额外的 O(n) 空间
    for (int i = 0; i < nums.size(); ++i) {
        long long complement = static_cast<long long>(target) - nums[i];
        // C++20 写法：contains；若是 C++17 可改成 seen.find(complement) != seen.end()
        if (seen.contains(complement)) {  // C++20
            return {seen[complement], i};
        }
        seen[nums[i]] = i;
    }
    return {-1, -1};
}

|      | 暴力法 | 哈希表法 |
| ---- | ------ | -------- |
| 时间 | O(n²)  | 期望 O(n)，最坏 O(n²) |
| 空间 | O(1)   | O(n)     |
| 权衡 | 省空间 | 期望情况下省时间 |

> **在实际工程中，通常优先优化时间**，因为内存相对廉价，而用户的等待时间是宝贵的。但在嵌入式系统等内存受限的环境中，空间可能更重要。

---

## 4.7 从代码分析时间与空间复杂度（Time and Space Complexity from Code）

这是将理论落地的关键一节。你将学会看着一段代码，就能快速判断它的时间和空间复杂度。

### 4.7.1 分析时间复杂度的系统方法

#### 规则1：顺序语句——取较大者

In [ ]:
void example(int n) {
    // 第一段: O(n)
    for (int i = 0; i < n; ++i) {
        std::cout << i << " ";
    }

    // 第二段: O(n²)
    for (int i = 0; i < n; ++i) {
        for (int j = 0; j < n; ++j) {
            std::cout << i * j << " ";
        }
    }
}
// 总时间: O(n) + O(n²) = O(n²)
// 规则：O(n) 被 O(n²) 支配，只保留最大的

#### 规则2：嵌套循环——相乘

In [ ]:
void nested_example(int n) {
    for (int i = 0; i < n; ++i) {           // 外层: n 次
        for (int j = 0; j < n; ++j) {       // 内层: n 次
            for (int k = 0; k < n; ++k) {   // 最内层: n 次
                std::cout << i + j + k;     // O(1), 基本操作
            }
        }
    }
}
// 总时间: n × n × n = O(n³)

#### 规则3：内层循环次数依赖于外层变量

In [ ]:
void dependent_loops(int n) {
    for (int i = 0; i < n; ++i) {
        for (int j = 0; j < i; ++j) {   // 注意：j < i，不是 j < n
            std::cout << "*";
        }
        std::cout << std::endl;
    }
}
// i = 0: 内层执行 0 次
// i = 1: 内层执行 1 次
// i = 2: 内层执行 2 次
// ...
// i = n-1: 内层执行 n-1 次
// 总次数 = 0 + 1 + 2 + ... + (n-1) = n(n-1)/2 = O(n²)

#### 规则4：循环变量以倍数增长——O(log n)

In [ ]:
void log_example(int n) {
    int i = 1;
    while (i < n) {
        std::cout << i << " ";
        i *= 2;     // 每次翻倍: 1, 2, 4, 8, 16, ...
    }
}
// i 的变化: 1 → 2 → 4 → 8 → ... → n
// 执行次数: 2^k = n → k = log₂(n)
// 时间: O(log n)

In [ ]:
void another_log_example(int n) {
    for (int i = n; i >= 1; i /= 2) {   // 每次减半
        std::cout << i << " ";
    }
}
// i 的变化: n → n/2 → n/4 → ... → 1
// 执行次数: log₂(n)
// 时间: O(log n)

#### 规则5：条件语句——取最坏情况

In [ ]:
void conditional_example(const std::vector<int>& arr, int x) {
    if (x > 0) {
        // O(n) 操作
        for (int i = 0; i < arr.size(); ++i) {
            std::cout << arr[i];
        }
    } else {
        // O(n²) 操作
        for (int i = 0; i < arr.size(); ++i) {
            for (int j = 0; j < arr.size(); ++j) {
                std::cout << arr[i] + arr[j];
            }
        }
    }
}
// 最坏情况: O(n²)（取两个分支中较大的）

#### 规则6：函数调用——代入分析

In [ ]:
void inner_function(int n) {   // 这个函数是 O(n)
    for (int i = 0; i < n; ++i) {
        std::cout << i;
    }
}

In [ ]:
void outer_function(int n) {
    for (int i = 0; i < n; ++i) {    // 外层循环 n 次
        inner_function(n);            // 每次调用 O(n)
    }
}
// 总时间: n × O(n) = O(n²)

### 4.7.2 分析空间复杂度的方法

空间复杂度分析需要找出：**除了输入数据之外，额外使用了多少内存？**

#### 情况1：只用了几个变量 → O(1)

In [ ]:
int find_max(const std::vector<int>& arr) {
    // 前提：arr 非空
    int max_val = arr[0];    // 1个额外变量
    for (int i = 1; i < arr.size(); ++i) {  // 1个额外变量 i
        if (arr[i] > max_val) {
            max_val = arr[i];
        }
    }
    return max_val;
}
// 额外空间: max_val + i = 2个变量 = O(1)
// 不管数组多大，额外空间始终是固定的

#### 情况2：创建了与输入同规模的数据结构 → O(n)

In [ ]:
std::vector<int> reverse_array(const std::vector<int>& arr) {
    int n = arr.size();
    std::vector<int> result(n);         // 创建了大小为 n 的新数组
    for (int i = 0; i < n; ++i) {
        result[i] = arr[n - 1 - i];
    }
    return result;
}
// 额外空间: result 数组占 n 个 int → O(n)

对比 **原地反转**（O(1) 空间）：

In [ ]:
void reverse_in_place(std::vector<int>& arr) {
    int left = 0, right = arr.size() - 1;
    while (left < right) {
        std::swap(arr[left], arr[right]);   // 只用了 left、right 两个变量
        ++left;
        --right;
    }
}
// 额外空间: left + right = O(1)
// 没有创建新数组，直接在原数组上操作

#### 情况3：递归——别忘了调用栈的空间

递归是空间分析中最容易被忽略的部分。每一层递归都会在栈上创建一个栈帧：

In [ ]:
int factorial(int n) {
    if (n <= 1) return 1;
    return n * factorial(n - 1);
}
// 递归深度为 n:
// factorial(5) → factorial(4) → factorial(3) → factorial(2) → factorial(1)
// 同时有 5 个栈帧存在
// 空间: O(n)

对比 **迭代版本**（O(1) 空间）：

In [ ]:
int factorial_iterative(int n) {
    int result = 1;
    for (int i = 2; i <= n; ++i) {
        result *= i;
    }
    return result;
}
// 只用了 result 和 i → O(1)

**递归空间分析的关键公式**：

```
递归的空间复杂度 = O(最大递归深度)
```

In [ ]:
// 二分查找（递归版）

In [ ]:
int binary_search_recursive(const std::vector<int>& arr, int target,
                             int left, int right) {
    if (left > right) return -1;
    int mid = left + (right - left) / 2;
    if (arr[mid] == target) return mid;
    if (arr[mid] < target)
        return binary_search_recursive(arr, target, mid + 1, right);
    else
        return binary_search_recursive(arr, target, left, mid - 1);
}
// 每次递归，搜索范围减半
// 最大递归深度: log₂(n)
// 空间: O(log n)

#### 情况4：二维数据结构 → O(n²)

In [ ]:
void create_distance_matrix(int n) {
    // 创建 n×n 的邻接矩阵
    std::vector<std::vector<int>> matrix(n, std::vector<int>(n, 0));
    
    for (int i = 0; i < n; ++i) {
        for (int j = 0; j < n; ++j) {
            matrix[i][j] = std::abs(i - j);
        }
    }
}
// 额外空间: n × n = O(n²)

### 4.7.3 综合练习：从代码分析复杂度

#### 练习1：分析下面代码的时间和空间复杂度

In [ ]:
void mystery1(int n) {
    for (int i = 1; i <= n; i *= 2) {      // log(n) 次
        for (int j = 0; j < n; ++j) {      // n 次
            std::cout << i + j << " ";
        }
    }
}

**分析**：
- 外层: `i` 从 1 倍增到 n → 执行 `log₂(n)` 次
- 内层: 每次执行 `n` 次
- 时间: O(n log n)
- 空间: O(1)（只用了 `i`, `j` 两个变量）

#### 练习2

In [ ]:
void mystery2(int n) {
    int count = 0;
    for (int i = 0; i < n; ++i) {
        for (int j = i; j < n; ++j) {   // 注意 j 从 i 开始
            ++count;
        }
    }
    std::cout << count << std::endl;
}

**分析**：
```
i = 0: 内层执行 n 次
i = 1: 内层执行 n-1 次
i = 2: 内层执行 n-2 次
...
i = n-1: 内层执行 1 次

总次数 = n + (n-1) + (n-2) + ... + 1 = n(n+1)/2
```
- 时间: O(n²)
- 空间: O(1)

#### 练习3

In [ ]:
std::vector<std::vector<int>> mystery3(const std::vector<int>& arr) {
    int n = arr.size();
    std::vector<std::vector<int>> result;
    
    for (int i = 0; i < n; ++i) {
        for (int j = i; j < n; ++j) {
            std::vector<int> sub(arr.begin() + i, arr.begin() + j + 1);
            result.push_back(sub);
        }
    }
    return result;
}

**分析**：
- 这段代码生成了所有连续子数组
- 子数组总数为 n(n+1)/2 个 → 外层循环 O(n²)
- 每个子数组的平均长度为 O(n)（创建 `sub` 并 push_back）
- 时间: O(n³)
- 空间: result 中存储了总共 O(n³) 个元素 → O(n³)

#### 练习4：递归复杂度

In [ ]:
int fibonacci(int n) {
    if (n <= 1) return n;
    return fibonacci(n - 1) + fibonacci(n - 2);
}

**分析**：

```
                fib(5)
              /        \
          fib(4)        fib(3)
         /    \         /    \
      fib(3)  fib(2)  fib(2) fib(1)
      /  \    /  \     / \
   fib(2) fib(1) ...  ...
   / \
fib(1) fib(0)
```

- 每个调用分裂成 2 个子调用，树的深度为 n
- 时间: O(φⁿ)（常用上界写作 O(2ⁿ)）← **指数级，非常慢！**
- 空间: O(n) ← 最大递归深度为 n（调用栈上同时最多有 n 个栈帧）

> 注意时间和空间不同：虽然总共有 2ⁿ 个调用，但它们不是同时存在的。某条分支返回后栈帧就释放了，所以栈的最大深度只有 n。

### 4.7.4 常见算法复杂度速查表

| 算法                 | 时间复杂度（平均） | 时间复杂度（最坏） | 单次操作辅助空间（不含输入和预先建好的结构） |
| -------------------- | ------------------ | ------------------ | ---------------------------------------------- |
| 数组随机访问         | O(1)               | O(1)               | O(1)                                           |
| 线性查找             | O(n)               | O(n)               | O(1)                                           |
| 二分查找（迭代）     | O(log n)           | O(log n)           | O(1)                                           |
| 二分查找（递归）     | O(log n)           | O(log n)           | O(log n)                                       |
| 冒泡排序             | O(n²)              | O(n²)              | O(1)                                           |
| 插入排序             | O(n²)              | O(n²)              | O(1)                                           |
| 归并排序             | O(n log n)         | O(n log n)         | O(n)                                           |
| 快速排序             | O(n log n)         | O(n²)              | 平均 O(log n)，最坏 O(n)                       |
| 哈希表查找           | O(1)               | O(n)               | O(1)；哈希表本身通常占 O(n)                    |
| 二叉搜索树查找（迭代） | O(log n)         | O(n)               | O(1)；树本身占 O(n)                            |

### 4.7.5 复杂度分析的实用口诀

```
看到单层循环 0 到 n          → O(n)
看到双层嵌套循环              → 很可能 O(n²)（具体看内层range）
看到循环变量每次乘以 2 或除以 2 → O(log n)
看到递归一分为二 + 线性合并    → O(n log n)
看到递归两次调用自身           → 很可能 O(2ⁿ)
看到排列组合 / 全排列          → O(n!)
```

---

## 本章总结

```
第四章知识体系全览
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

4.1 导论
  └── 数据结构 = 数据的组织方式
  └── 程序 = 数据结构 + 算法

4.2-4.3 栈内存与堆内存
  └── 栈: 系统管理、快速、小、LIFO
  └── 堆: 手动管理、灵活、大、需要 new/delete
  └── 现代C++: 用智能指针和容器避免手动管理

4.4 物理结构 vs 逻辑结构
  └── 物理: 数组（连续）、链表（非连续）
  └── 逻辑: 栈、队列、树、图、哈希表...
  └── 逻辑结构建立在物理结构之上

4.5 抽象数据类型（ADT）
  └── ADT = 数据 + 操作（只定义接口，不定义实现）
  └── 实现了接口与实现的分离

4.6-4.7 时间与空间复杂度
  └── 大O表示法: 描述增长趋势的上界
  └── O(1) < O(log n) < O(n) < O(n log n) < O(n²) < O(2ⁿ)
  └── 6条代码分析规则: 顺序取大、嵌套相乘、倍增取对数...
  └── 时空权衡: 空间换时间是常用策略
```

---